In [ ]:
import base64, requests # Import necessary libraries: base64 for encoding, requests for making HTTP requests.
from config import HF_API_KEY # Import the Hugging Face API key from a local config file.

API_URL = "https://router.huggingface.co/v1/chat/completions" # Define the API endpoint for chat completions.
H EADERS = {"Authorization": f"Bearer {HF_API_KEY}", "Content-Type": "application/json"} # Set up authorization headers with the API key.
MODELS = [ # Define a list of AI models to be used for image captioning.
    "zai-org/GLM-4.5V",
    "Qwen/Qwen2.5-VL-72B-Instruct",
    "Qwen/Qwen2.5-VL-32B-Instruct",
    "google/gemma-3-27b-it",
]

def data_url(b: bytes) -> str:
    return "data:image/jpeg;base64," + base64.b64encode(b).decode("utf-8") # Convert image bytes to a base64 data URL for API submission.

def extract_err(r: requests.Response) -> str:
    try: # Attempt to parse the error message from a JSON response.
        j = r.json()
        return j.get("error", {}).get("message") or str(j) # Extract specific error message or return the full JSON.
    except Exception:
        return (r.text or "").strip() or r.reason or "Request failed." # Fallback to raw text or generic message if JSON parsing fails.

def box(title: str, lines: list[str], icon: str):
    w = max(30, len(title) + 4, *(len(x) for x in lines)) # Calculate the width for the box, ensuring it's wide enough for content.
    print("\n" + "┏" + "━" * (w + 2) + "┓") # Print the top border of the box.
    print(f"┃ {icon} {title.ljust(w - 2)} ┃") # Print the title with an icon, left-justified.
    print("┣" + "━" * (w + 2) + "┫") # Print the separator line.
    for x in lines:
        print(f"┃ {x.ljust(w)} ┃") # Print each line of content, left-justified.
    print("┗" + "━" * (w + 2) + "┛\n") # Print the bottom border of the box.

def caption_single_image():
    image_source = input("🖼️ Enter image filename (default: test.jpg): ").strip() or "test.jpg" # Prompt user for image filename, defaulting to 'test.jpg'.
    try:
        with open(image_source, "rb") as f:
            img = f.read() # Read the image file in binary mode.
    except Exception as e:
        box("File Error", [f"Could not load: {image_source}", f"Reason: {e}"], "❌") # Display an error box if the file cannot be loaded.
        return

    base = { # Define the base structure for the API request payload.
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": "Give a short caption for this image."},
                {"type": "image_url", "image_url": {"url": data_url(img)}},
            ],
        }],
        "max_tokens": 60,
        "temperature": 0.2,
    }

    last = None # Initialize variable to store the last error message.
    for model in MODELS: # Iterate through each defined AI model.
        payload = dict(base, model=model) # Create a payload specific to the current model.
        try:
            r = requests.post(API_URL, headers=HEADERS, json=payload, timeout=120) # Send a POST request to the API with a timeout.
        except requests.RequestException as e:
            last = f"Request failed: {e}" # Store request-specific errors.
            continue # Continue to the next model if the request fails.

        if r.status_code != 200:
            last = extract_err(r) # Extract and store error message if the HTTP status code is not 200.
            continue # Continue to the next model.

        try:
            d = r.json() # Attempt to parse the JSON response.
        except Exception:
            last = "Non-JSON response received from the API." # Store error if response is not valid JSON.
            continue # Continue to the next model.

        cap = (d.get("choices", [{}])[0].get("message", {}).get("content") or "").strip() # Extract the caption from the API response.
        if cap:
            box("Image Caption Generated", [ # If a caption is found, display it in a success box.
                f"🖼️ Image  : {image_source}",
                "📝 Caption:",
                f"   {cap}",
            ], "🎉")
            return # Exit the function after a successful caption.
        last = "No caption found." # If no caption is extracted, store a 'no caption' message.

    box("Caption Failed", [f"🖼️ Image  : {image_source}", f"❌ Error : {last or 'Unknown error'}"], "⚠️") # If all models fail, display a failure box.

def main():
    caption_single_image() # Call the function to caption a single image.

if __name__ == "__main__":
    main() # Run the main function when the script is executed.

In [ ]:
import base64
import csv
import os

# --- Placeholder for your specific AI client library ---
# (Replace this mock function with your actual API call, e.g., google-genai or openai)
def call_ai_caption_api(image_base64_string):
    """
    Sends the base64 image data to the AI API.
    """
    # This matches the structure in your screenshot:
    payload = {
        "messages": {
            "role": "user",
            "content": [
                {"type": "text", "text": "Give a short caption for this image."},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64_string}"}}
            ]
        }
    }

    # --- Simulate API Response for demonstration ---
    # In reality, you would pass 'payload' to your AI SDK here.
    return "A beautiful space background with stars and nebulae."

# --- Helper Functions ---

def clean_file_path(path_input):
    """Cleans up user-input paths by stripping extra spaces and quotation marks."""
    return path_input.strip().strip('"\'')

def encode_image_to_base64(image_path):
    """Reads a local image file and converts it to a base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def initialize_report(report_path):
    """Creates a CSV report file with headers if it doesn't already exist."""
    if not os.path.exists(report_path):
        with open(report_path, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["Image Path", "Status", "Caption / Error Message"])

def append_to_report(report_path, image_path, status, message):
    """Appends a single processing result row to the CSV report."""
    with open(report_path, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([image_path, status, message])

# --- Main Program Workflow ---

def main():
    report_file = "caption_report.csv"
    initialize_report(report_file)

    print("=" * 60)
    print("        WELCOME TO THE MULTI-IMAGE CAPTIONING WORKFLOW")
    print("=" * 60)
    print(f"Results will be automatically tracked in: {os.path.abspath(report_file)}")
    print("Type 'exit' at any time to finish.\n")

    processed_count = 0

    while True:
        # 1. Prompt for user input
        user_input = input(f"[{processed_count + 1}] Enter image file path: ")

        # 2. Check for exit command
        if user_input.strip().lower() in ['exit', 'quit', 'done']:
            print("\nExiting workflow...")
            break

        # 3. Clean and validate path input
        image_path = clean_file_path(user_input)

        if not image_path:
            print("❌ Input cannot be empty. Please try again.\n")
            continue

        print(f"Processing: {os.path.basename(image_path)}...")

        # 4. Error Handling & Processing
        try:
            # Convert image to base64 string
            base64_data = encode_image_to_base64(image_path)

            # Generate caption via AI function
            caption = call_ai_caption_api(base64_data)

            # Log success to console and report
            print(f"✨ Caption: {caption}\n")
            append_to_report(report_file, image_path, "Success", caption)
            processed_count += 1

        except FileNotFoundError:
            error_msg = "File not found. Verify the path is correct."
            print(f"❌ Error: {error_msg}\n")
            append_to_report(report_file, image_path, "Failed", error_msg)

        except Exception as e:
            error_msg = f"Unexpected error: {str(e)}"
            print(f"❌ Error: {error_msg}\n")
            append_to_report(report_file, image_path, "Failed", error_msg)

    # Final Summary
    print("=" * 60)
    print(f"Session finished! Successfully processed {processed_count} images.")
    print("=" * 60)

if __name__ == "__main__":
    main()
